# 2a. ERA5: model evaluation and observed change (calculations)

Two questions, asked of ERA5 and the LESFMIP ensembles over the years they share (1979–2013):

1. **Is each model consistent with ERA5?** Could ERA5 be one more member of a model's historical ensemble? (Part 1)
2. **Is the observed change detectable, and which forcings is it consistent with?** (Part 2)

This notebook works everything out and saves it under `results/tas/<RUN>/`; notebook 02b opens the results and draws the figures.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from extant import datatree, era5_evaluation as ev, observed_change as oc, paths, preprocessing as pp, storage
from extant.datatree import skip_empty, tree_to_dataset
from extant.logger import setup_logging
from extant.plots import methods

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Dask warns about every large task graph; the graphs here are large on purpose
warnings.filterwarnings('ignore', message='.*Sending large graph of size.*', category=UserWarning)

## Settings

In [ ]:
#(t): The variable, and its units once notebook 01 has converted it
VARIABLE = 'tas'
UNITS = '°C'

#(t): The experiments analysed (notebook 01's), and the seasons in the order of every figure
EXPERIMENTS = ['hist-nat', 'hist-aer', 'hist-totalO3', 'hist-GHG', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

In [ ]:
#(t): The years ERA5 and LESFMIP share: ERA5 starts in 1979, and LESFMIP's last season is DJF 2013
YEARS = slice(1979, 2013)

In [ ]:
#(t): The grid point of the tests at one point (the 2.5° cell nearest config.POINT, on the Antarctic plateau)
POINT = dict(lat=-82.5, lon=120.0)

#(t): The model and season of the one-model, one-season quick looks
MODEL = 'CanESM5'
SEASON = 'DJF'

In [ ]:
#(t): Where this notebook saves a result: results/tas/<RUN>/<name>.zarr
def result_path(name):
    return paths.result(name, VARIABLE, RUN)

## Open the data

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (seasonal means, /<model>/<experiment>, on member × year × season × lat × lon), `era5_seasonal_da` (ERA5 on year × season × lat × lon, on the same grid) and `RUN`, all in °C.

### Sample data

The 3×3 block of grid cells around the point in `tests/data` (two models), put through notebook 01's pre-processing.

In [ ]:
#(t): The sample's files: <model>_<experiment>_sample.nc, monthly tas in K on (member, time, lat, lon)
sample_files = sorted(paths.SAMPLE_DIR.glob('*_*_sample.nc'))
sample_files

In [ ]:
#(t): The model and experiment of each file, from its name
sample_names = [path.stem.removesuffix('_sample').rsplit('_', 1) for path in sample_files]
sample_names

In [ ]:
#(t): Each file of the experiments analysed opened, named /<model>/<experiment>
lesfmip_monthly_datasets = {f'{model}/{experiment}': xr.open_dataset(path).load()
                            for path, (model, experiment) in zip(sample_files, sample_names) if experiment in EXPERIMENTS}
lesfmip_monthly_tree = xr.DataTree.from_dict(lesfmip_monthly_datasets)
lesfmip_monthly_tree

In [ ]:
#(t): Seasonal means in °C, through notebook 01's pre-processing (pp.seasonal_tree: to °C, cut at 2014-12, complete
#(t): seasons in whole years)
lesfmip_season_tree = pp.seasonal_tree(lesfmip_monthly_tree, VARIABLE).compute()
lesfmip_season_tree

In [ ]:
#(t): ERA5's sample file: monthly tas in K from 1979, on (time, lat, lon)
era5_sample_path = paths.SAMPLE_DIR / 'era5_sample.nc'
era5_monthly_da = xr.open_dataarray(era5_sample_path).load()
era5_monthly_da

In [ ]:
#(t): Its seasonal means in °C, on the models' grid (pp.seasonal_era5, as in notebook 01)
era5_seasonal_da = pp.seasonal_era5(era5_monthly_da, VARIABLE, like=lesfmip_season_tree)
RUN = 'sample'

### Full data (JASMIN)

The seasonal means notebook 01 saved (section 3), read from `paths.DATA_DIR` if they have been moved there, otherwise from `paths.SCRATCH`. Opening is lazy.

In [ ]:
#(t): The two stores' paths
seasonal_path = paths.find(paths.seasonal('lesfmip', VARIABLE))
era5_path = paths.find(paths.seasonal('era5', VARIABLE))
seasonal_path, era5_path

In [ ]:
#(t): Every model and experiment notebook 01 saved
lesfmip_saved_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})

In [ ]:
#(t): Only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = lesfmip_saved_tree.filter(lambda node: node.name in EXPERIMENTS)
lesfmip_season_tree

In [ ]:
era5_seasonal_da = xr.open_zarr(era5_path)[VARIABLE].load()
RUN = 'full'

### What was opened

Whichever section ran. An empty square is a model and experiment that was never opened.

In [ ]:
#(t): Members per model (rows) and experiment (columns)
member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member']
                              for node in lesfmip_season_tree.leaves}).unstack()
member_counts_df = member_counts_df.reindex(columns=EXPERIMENTS)
fig, ax = methods.member_count_heatmap(member_counts_df, title='LESFMIP members per model and experiment')

In [ ]:
#(t): ERA5's seasonal means
era5_seasonal_da

In [ ]:
#(t): Quick look: ERA5 at the point, every season
era5_seasonal_da.sel(POINT).plot.line(x='year', hue='season');

In [ ]:
#(t): Each experiment's members at the point (one model, DJF)
fig, axes = methods.members_by_experiment(lesfmip_season_tree[MODEL].sel(**POINT).load(),
                                          EXPERIMENTS, VARIABLE, units=UNITS)

## A first look

ERA5's 1979–2013 mean next to each model's historical ensemble mean, in one season: the mean-state bias, which only the tests of raw values see; every other test removes it.

In [ ]:
#(t): ERA5's 1979-2014 mean in SEASON, on (lat, lon)
era5_mean_da = era5_seasonal_da.sel(year=YEARS, season=SEASON).mean('year')
era5_mean_da

In [ ]:
#(t): Each model's historical members over the same years, in the same season
first_look_tree = lesfmip_season_tree.match('*/historical').sel(year=YEARS, season=SEASON)
first_look_tree

In [ ]:
#(t): Each model's ensemble mean over those years
first_look_mean_tree = first_look_tree.mean(['year', 'member'])

In [ ]:
#(t): In one DataArray, on (model, lat, lon)
first_look_mean_ds = datatree.tree_to_dataset(first_look_mean_tree, dims=['model', 'experiment']).load()
historical_mean_da = first_look_mean_ds[VARIABLE].sel(experiment='historical', drop=True)
historical_mean_da

In [ ]:
#(t): Each model's bias: its ensemble mean minus ERA5's
model_bias_da = historical_mean_da - era5_mean_da
model_bias_da

In [ ]:
#(t): Quick look: each model's bias on the plain lat-lon grid
model_bias_da.plot(x='lon', y='lat', col='model', cmap='RdBu_r', center=0);

In [ ]:
#(t): Saved for 02b's polar maps
storage.save(xr.Dataset({'era5_mean': era5_mean_da, 'model_bias': model_bias_da}), result_path('first_look'))

## What the data look like: climatologies

Every season: the 1979–2013 average (the climatology) and the year-to-year spread of ERA5 and of each model's historical ensemble, how they differ, the area mean through time, and one model's experiments.

### ERA5

In [ ]:
#(t): ERA5's climatology: its mean over YEARS in each season, on (season, lat, lon)
era5_climatology_da = era5_seasonal_da.sel(year=YEARS, season=SEASONS).mean('year')
era5_climatology_da

In [ ]:
era5_climatology_da.plot(x='lon', y='lat', col='season', cmap='RdBu_r');

In [ ]:
#(t): ERA5's year-to-year spread: the standard deviation of its seasonal means over YEARS
era5_std_da = era5_seasonal_da.sel(year=YEARS, season=SEASONS).std('year')
era5_std_da

In [ ]:
era5_std_da.plot(x='lon', y='lat', col='season', cmap='Reds');

In [ ]:
#(t): ERA5 south of 60°S
era5_south_da = era5_seasonal_da.sel(season=SEASONS, lat=slice(None, -60))

In [ ]:
#(t): its area mean in every year (cos-latitude weighted)
era5_area_weights_da = np.cos(np.deg2rad(era5_south_da.lat))
era5_area_mean_da = era5_south_da.weighted(era5_area_weights_da).mean(['lat', 'lon'])
era5_area_mean_da

In [ ]:
era5_area_mean_da.plot(x='year', hue='season');

### The models' historical ensembles

In [ ]:
#(t): Each model's historical ensemble
historical_tree = lesfmip_season_tree.match('*/historical')
historical_tree

In [ ]:
#(t): over YEARS, in the seasons' order
historical_eval_tree = historical_tree.sel(year=YEARS, season=SEASONS)

In [ ]:
#(t): Each model's climatology: the mean over the years and the members
historical_climatology_tree = historical_eval_tree.mean(['year', 'member'])

In [ ]:
%%time
#(t): In one DataArray, on (model, season, lat, lon)
historical_climatology_ds = datatree.tree_to_dataset(historical_climatology_tree, dims=['model', 'experiment']).load()
historical_climatology_da = historical_climatology_ds[VARIABLE].sel(experiment='historical', drop=True)
historical_climatology_da

In [ ]:
historical_climatology_da.plot(x='lon', y='lat', row='model', col='season', cmap='RdBu_r');

In [ ]:
#(t): Each model's climatology minus ERA5's
historical_bias_da = historical_climatology_da - era5_climatology_da
historical_bias_da

In [ ]:
historical_bias_da.plot(x='lon', y='lat', row='model', col='season', cmap='RdBu_r', center=0);

In [ ]:
#(t): Each member's year-to-year standard deviation
historical_member_std_tree = historical_eval_tree.std('year')

In [ ]:
#(t): averaged over the members
historical_std_tree = historical_member_std_tree.mean('member')

In [ ]:
%%time
#(t): In one DataArray, on (model, season, lat, lon)
historical_std_ds = datatree.tree_to_dataset(historical_std_tree, dims=['model', 'experiment']).load()
historical_std_da = historical_std_ds[VARIABLE].sel(experiment='historical', drop=True)
historical_std_da

In [ ]:
historical_std_da.plot(x='lon', y='lat', row='model', col='season', cmap='Reds');

In [ ]:
#(t): Each model's spread over ERA5's: above 1, the model varies more from year to year than ERA5
historical_std_ratio_da = historical_std_da / era5_std_da
historical_std_ratio_da

In [ ]:
historical_std_ratio_da.plot(x='lon', y='lat', row='model', col='season', cmap='RdBu_r', center=1);

In [ ]:
#(t): Each model's historical ensemble mean, every year
historical_ensemble_mean_tree = historical_tree.mean('member')

In [ ]:
#(t): In one DataArray, on (model, year, season, lat, lon)
historical_ensemble_mean_ds = datatree.tree_to_dataset(historical_ensemble_mean_tree, dims=['model', 'experiment'])
historical_ensemble_mean_da = historical_ensemble_mean_ds[VARIABLE].sel(experiment='historical', drop=True)
historical_ensemble_mean_da

In [ ]:
#(t): south of 60°S, in the seasons' order
historical_south_da = historical_ensemble_mean_da.sel(season=SEASONS, lat=slice(None, -60))

In [ ]:
%%time
#(t): its area mean in every year (cos-latitude weighted)
historical_area_weights_da = np.cos(np.deg2rad(historical_south_da.lat))
historical_area_mean_da = historical_south_da.weighted(historical_area_weights_da).mean(['lat', 'lon']).load()
historical_area_mean_da

In [ ]:
#(t): The models' ensemble means (colours) and ERA5 (black), one panel per season
fg = historical_area_mean_da.plot(x='year', hue='model', col='season')
for ax, season in zip(fg.axs.flat, SEASONS):
    era5_area_mean_da.sel(season=season).plot(ax=ax, color='k')

### One model's experiments

In [ ]:
#(t): MODEL's every experiment, over YEARS, in the seasons' order
model_eval_tree = lesfmip_season_tree[MODEL].sel(year=YEARS, season=SEASONS)
model_eval_tree

In [ ]:
#(t): Each experiment's climatology: the mean over the years and the members
model_climatology_tree = model_eval_tree.mean(['year', 'member'])

In [ ]:
%%time
#(t): In one DataArray, on (experiment, season, lat, lon)
model_climatology_ds = datatree.tree_to_dataset(model_climatology_tree, dims=['experiment']).load()
model_climatology_da = model_climatology_ds[VARIABLE]
model_climatology_da

In [ ]:
model_climatology_da.plot(x='lon', y='lat', row='experiment', col='season', cmap='RdBu_r');

In [ ]:
#(t): Each experiment's climatology minus hist-nat's, over the same years
model_difference_da = model_climatology_da - model_climatology_da.sel(experiment='hist-nat')
model_difference_da

In [ ]:
model_difference_da.plot(x='lon', y='lat', row='experiment', col='season', cmap='RdBu_r', center=0);

# Part 1: is each model consistent with ERA5?

Every test asks one question: *could ERA5 be one more member of the historical ensemble?* If a model simulates both the forced response (the **signal**) and the internal variability (the **noise**) correctly, ERA5 is statistically indistinguishable from any one member. So each test computes the same statistic for ERA5 and for every member, and asks where ERA5 falls among the members (Suarez-Gutierrez, Milinski & Maher, 2021, *Clim. Dyn.*, [doi:10.1007/s00382-021-05821-w](https://doi.org/10.1007/s00382-021-05821-w)). The members are the null distribution, so no shape is assumed, and each member is exactly as long as ERA5.

First at one grid point, then on the maps (section 7 and 8) and in regional means (section 9).

## The data at the point

Each model's historical members and ERA5 over `YEARS`. The models stay in a DataTree, each with its own members; ERA5 is one DataArray.

In [ ]:
#(t): LESFMIP over the years shared with ERA5
lesfmip_obs_tree = lesfmip_season_tree.sel(year=YEARS)

In [ ]:
#(t): Each model's historical members at the point, on (member, year, season)
historical_point_tree = lesfmip_obs_tree.match('*/historical').sel(POINT).load()
historical_point_tree

In [ ]:
#(t): ERA5 at the point over the same years, on (year, season)
era5_point_da = era5_seasonal_da.sel(year=YEARS).sel(POINT)
era5_point_da

In [ ]:
#(t): Check: every model has the same years as ERA5, and nothing is missing (the tests compare the same samples)
{node.parent.name: (int(node.year.min()), int(node.year.max()), int(node['tas'].isnull().sum()))
 for node in historical_point_tree.leaves}, (int(era5_point_da.year.min()), int(era5_point_da.year.max()),
                                             int(era5_point_da.isnull().sum()))

In [ ]:
#(t): Quick look: MODEL's members (grey) and ERA5 (black), every season
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharex=True)
for ax, season in zip(axes, SEASONS):
    historical_point_tree[MODEL]['historical']['tas'].sel(season=season).plot.line(ax=ax, x='year', hue='member',
                                                                                    add_legend=False, color='0.7', lw=0.5)
    era5_point_da.sel(season=season).plot(ax=ax, color='k')
    ax.set_title(season)

In [ ]:
#(t): Each model's hist-nat members at the point too, as context for 02b's time series
histnat_point_tree = lesfmip_obs_tree.match('*/hist-nat').sel(POINT).load()

In [ ]:
storage.save(historical_point_tree, result_path('historical_point'))

In [ ]:
storage.save(histnat_point_tree, result_path('histnat_point'))

In [ ]:
storage.save(era5_point_da.rename('tas'), result_path('era5_point'))

## The three treatments

Each series (every member and ERA5, separately, in each season) is treated in the same way, so they all lose the same things with the same sampling noise:

| Treatment | Removed from each series | A test of it then sees |
|---|---|---|
| raw values | nothing | the climatology; a mean-state bias dominates |
| anomalies | its own mean over the years | the forced trend (signal) and the variability (noise) |
| detrended | its own least-squares line | the variability (noise) only |

In [ ]:
#(t): Anomalies: each member minus its own mean over the years
historical_anomaly_tree = historical_point_tree - historical_point_tree.mean('year')

In [ ]:
#(t): ... and ERA5 minus its own
era5_anomaly_da = era5_point_da - era5_point_da.mean('year')

In [ ]:
#(t): Detrended: each member minus its own least-squares line through the years (ev.detrend)
historical_detrended_tree = historical_point_tree.map_over_datasets(skip_empty(ev.detrend))

In [ ]:
#(t): ... and ERA5 minus its own
era5_detrended_da = ev.detrend(era5_point_da)

In [ ]:
#(t): Quick look: ERA5 (black) and MODEL's first member (grey) in SEASON: raw, anomalies, detrended
member_da = historical_point_tree[MODEL]['historical']['tas'].isel(member=0).sel(season=SEASON)
anomaly_da = historical_anomaly_tree[MODEL]['historical']['tas'].isel(member=0).sel(season=SEASON)
detrended_da = historical_detrended_tree[MODEL]['historical']['tas'].isel(member=0).sel(season=SEASON)
fig, axes = plt.subplots(1, 3, figsize=(16, 3), sharex=True)
member_da.plot(ax=axes[0], color='0.6')
era5_point_da.sel(season=SEASON).plot(ax=axes[0], color='k')
anomaly_da.plot(ax=axes[1], color='0.6')
era5_anomaly_da.sel(season=SEASON).plot(ax=axes[1], color='k')
detrended_da.plot(ax=axes[2], color='0.6')
era5_detrended_da.sel(season=SEASON).plot(ax=axes[2], color='k');

## Where ERA5 falls among the members

Every test below ends the same way: the statistic of every member and ERA5's go to `ev.locate`, which returns for each season ERA5's value, the members' values and their 5%, 50% and 95%, ERA5's **percentile** among them (ties count half), the two-sided **p-value** (twice the smaller tail, ERA5 counted as one more member: 2 × (members at or beyond ERA5 on that side + 1) / (N + 1)), the **verdict** (−1 ERA5 below the members, +1 above, 0 not flagged; flagged when p < 0.1, i.e. ERA5 outside the members' 5–95%) and whether the model has enough members to flag anything (**testable**: N ≥ 20).

In [ ]:
#(t): ev.locate on one model's node: its members' statistic, and ERA5's
@skip_empty
def locate_era5(ds, era5_da):
    return ev.locate(ds['tas'], era5_da)

## The moment test

The first four moments of each series: the **mean** of the raw values, and the **standard deviation**, **skewness** and **excess kurtosis** of the detrended values, so the 1979–2013 warming neither widens the spread nor bends the shape. To test a moment on other values, start its first cell from another tree (`historical_point_tree`, `historical_anomaly_tree` or `historical_detrended_tree`, and the matching ERA5), and change its treatment where the four are put together.

### Mean (raw values)

In [ ]:
#(t): Each member's mean over the years, on (member, season)
historical_mean_tree = historical_point_tree.mean('year')
historical_mean_tree

In [ ]:
#(t): ERA5's
era5_mean_da = era5_point_da.mean('year')
era5_mean_da

In [ ]:
#(t): Quick look: MODEL's members (histogram) and ERA5 (black) in SEASON
historical_mean_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_mean_da.sel(season=SEASON)), color='k');

In [ ]:
#(t): Where ERA5's falls among each model's members
mean_test_tree = historical_mean_tree.map_over_datasets(locate_era5, era5_mean_da)
mean_test_tree

### Standard deviation (detrended)

In [ ]:
historical_std_tree = historical_detrended_tree.std('year', ddof=1)
era5_std_da = era5_detrended_da.std('year', ddof=1)

In [ ]:
historical_std_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_std_da.sel(season=SEASON)), color='k');

In [ ]:
std_test_tree = historical_std_tree.map_over_datasets(locate_era5, era5_std_da)

### Skewness (detrended)

In [ ]:
#(t): ev.skewness: the plain moment estimator; > 0 when the warm tail is longer
historical_skewness_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.skewness))
era5_skewness_da = ev.skewness(era5_detrended_da)

In [ ]:
historical_skewness_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_skewness_da.sel(season=SEASON)), color='k');

In [ ]:
skewness_test_tree = historical_skewness_tree.map_over_datasets(locate_era5, era5_skewness_da)

### Excess kurtosis (detrended)

In [ ]:
#(t): ev.excess_kurtosis: 0 for a normal distribution, > 0 for heavier tails
historical_kurtosis_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.excess_kurtosis))
era5_kurtosis_da = ev.excess_kurtosis(era5_detrended_da)

In [ ]:
historical_kurtosis_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_kurtosis_da.sel(season=SEASON)), color='k');

In [ ]:
kurtosis_test_tree = historical_kurtosis_tree.map_over_datasets(locate_era5, era5_kurtosis_da)

### The four together

In [ ]:
#(t): One node's four tests along a statistic dim, with the treatment each moment was computed on (for 02b's labels)
@skip_empty
def four_moments(mean_ds, std_ds, skewness_ds, kurtosis_ds):
    moments_ds = xr.concat([mean_ds, std_ds, skewness_ds, kurtosis_ds],
                           dim=xr.Variable('statistic', ['mean', 'std', 'skewness', 'kurtosis']))
    return moments_ds.assign_coords(treatment=('statistic', ['raw', 'detrended', 'detrended', 'detrended']))

In [ ]:
moment_test_tree = xr.map_over_datasets(four_moments, mean_test_tree, std_test_tree, skewness_test_tree,
                                        kurtosis_test_tree)
moment_test_tree

In [ ]:
storage.save(moment_test_tree, result_path('moment_test'))

## The other statistics

Not moments, but tested in exactly the same way:

| Statistic | Of | What it measures |
|---|---|---|
| trend | raw values | the least-squares trend per decade: the forced response (**signal**) |
| emergence | raw values | the S/N against itself: the change of a 25-year LOWESS curve from the start of the record to the end, over the std of the deviations from it |
| width | detrended | Q95 − Q05, whose forced change the response notebooks analyse (**noise**) |
| lower_tail, upper_tail | detrended | the cold tail Q50 − Q05 and the warm tail Q95 − Q50 |
| records_high, records_low | raw values | how many record highs and lows (1 + 1/2 + … + 1/35 ≈ 4.1 of each with no change) |
| lag1 | detrended | the lag-1 autocorrelation: the year-to-year memory |

### Trend

In [ ]:
#(t): The least-squares trend, per decade
historical_trend_tree = historical_point_tree.map_over_datasets(skip_empty(ev.trend))
era5_trend_da = ev.trend(era5_point_da)

In [ ]:
historical_trend_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_trend_da.sel(season=SEASON)), color='k');

In [ ]:
trend_test_tree = historical_trend_tree.map_over_datasets(locate_era5, era5_trend_da)

### Emergence

In [ ]:
#(t): The smoothed change over the record, over its own noise
historical_emergence_tree = historical_point_tree.map_over_datasets(skip_empty(ev.self_signal_to_noise))
era5_emergence_da = ev.self_signal_to_noise(era5_point_da)

In [ ]:
historical_emergence_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_emergence_da.sel(season=SEASON)), color='k');

In [ ]:
emergence_test_tree = historical_emergence_tree.map_over_datasets(locate_era5, era5_emergence_da)

### Width

In [ ]:
#(t): Q95 - Q05
historical_width_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.quantile_width))
era5_width_da = ev.quantile_width(era5_detrended_da)

In [ ]:
historical_width_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_width_da.sel(season=SEASON)), color='k');

In [ ]:
width_test_tree = historical_width_tree.map_over_datasets(locate_era5, era5_width_da)

### Lower tail

In [ ]:
#(t): Q50 - Q05
historical_lower_tail_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.lower_tail_width))
era5_lower_tail_da = ev.lower_tail_width(era5_detrended_da)

In [ ]:
historical_lower_tail_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_lower_tail_da.sel(season=SEASON)), color='k');

In [ ]:
lower_tail_test_tree = historical_lower_tail_tree.map_over_datasets(locate_era5, era5_lower_tail_da)

### Upper tail

In [ ]:
#(t): Q95 - Q50
historical_upper_tail_tree = historical_detrended_tree.map_over_datasets(skip_empty(ev.upper_tail_width))
era5_upper_tail_da = ev.upper_tail_width(era5_detrended_da)

In [ ]:
historical_upper_tail_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_upper_tail_da.sel(season=SEASON)), color='k');

In [ ]:
upper_tail_test_tree = historical_upper_tail_tree.map_over_datasets(locate_era5, era5_upper_tail_da)

### Records high

In [ ]:
#(t): Years warmer than every year before them
historical_records_high_tree = historical_point_tree.map_over_datasets(skip_empty(ev.record_highs))
era5_records_high_da = ev.record_highs(era5_point_da)

In [ ]:
historical_records_high_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_records_high_da.sel(season=SEASON)), color='k');

In [ ]:
records_high_test_tree = historical_records_high_tree.map_over_datasets(locate_era5, era5_records_high_da)

### Records low

In [ ]:
#(t): Years colder than every year before them
historical_records_low_tree = historical_point_tree.map_over_datasets(skip_empty(ev.record_lows))
era5_records_low_da = ev.record_lows(era5_point_da)

In [ ]:
historical_records_low_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_records_low_da.sel(season=SEASON)), color='k');

In [ ]:
records_low_test_tree = historical_records_low_tree.map_over_datasets(locate_era5, era5_records_low_da)

### Lag1

In [ ]:
#(t): ev.lag1_autocorrelation (xr.corr) takes one DataArray at a time, so it is applied to each variable of a node
@skip_empty
def lag1_autocorrelation(ds):
    return ds.map(ev.lag1_autocorrelation)

In [ ]:
#(t): The correlation of each year with the next
historical_lag1_tree = historical_detrended_tree.map_over_datasets(lag1_autocorrelation)
era5_lag1_da = ev.lag1_autocorrelation(era5_detrended_da)

In [ ]:
historical_lag1_tree[MODEL]['historical']['tas'].sel(season=SEASON).plot.hist(bins=20, color='0.7')
plt.axvline(float(era5_lag1_da.sel(season=SEASON)), color='k');

In [ ]:
lag1_test_tree = historical_lag1_tree.map_over_datasets(locate_era5, era5_lag1_da)

### All of them together

In [ ]:
#(t): One node's tests along a statistic dim, with the treatment of each (for 02b's labels)
@skip_empty
def other_statistics(*tests_ds):
    statistics_ds = xr.concat(tests_ds, dim=xr.Variable('statistic', ['trend', 'emergence', 'width', 'lower_tail',
                                                                     'upper_tail', 'records_high', 'records_low', 'lag1']))
    return statistics_ds.assign_coords(treatment=('statistic', ['raw', 'raw', 'detrended', 'detrended', 'detrended',
                                                                'raw', 'raw', 'detrended']))

In [ ]:
statistic_test_tree = xr.map_over_datasets(other_statistics, trend_test_tree, emergence_test_tree, width_test_tree,
                                           lower_tail_test_tree, upper_tail_test_tree, records_high_test_tree,
                                           records_low_test_tree, lag1_test_tree)
statistic_test_tree

In [ ]:
storage.save(statistic_test_tree, result_path('statistic_test'))

## The rank histograms

The test of Suarez-Gutierrez et al. (2021). Step by step on one model and season first, then `ev.rank_histogram_test` for every model.

1. Each year, ERA5's **rank**: how many members are below it.
2. The ranks of every year counted into a **histogram**. If ERA5 behaves like a member, every rank is equally likely and the histogram is flat.
3. With ~35 samples even a perfect model's histogram is not flat, so each member in turn is ranked against the other N − 1 (**pseudo-observations**): their histograms show how far from flat a perfect model's can be.
4. Each histogram summarised: the **dispersion** (the variance of the rank relative to a flat histogram: > 1 a U shape, too little spread; < 1 a dome, too much), how often ERA5 is outside every member, and the trend of the rank.
5. ERA5's summary located among the pseudo-observations'.

ERA5 is ranked against each of the N sub-ensembles that leave one member out, and those histograms averaged (`ev.leave_one_out_counts`), so it faces exactly the pseudo-observations' test.

In [ ]:
#(t): MODEL's anomalies and ERA5's in SEASON
season_members_da = historical_anomaly_tree[MODEL]['historical']['tas'].sel(season=SEASON)
season_era5_da = era5_anomaly_da.sel(season=SEASON)
season_members_da

In [ ]:
#(t): 1. ERA5's rank in each year (0 to N)
season_era5_rank_da = ev.ensemble_rank(season_members_da, season_era5_da)
season_era5_rank_da.plot(marker='o');

In [ ]:
#(t): 3. Each member's rank among the other N - 1, each year
season_pseudo_rank_da = ev.pseudo_observation_ranks(season_members_da)
season_pseudo_rank_da

In [ ]:
#(t): 2. and 3. The ranks counted over the years, on the same N bins
n_season_members = season_members_da.sizes['member']
season_era5_counts_da = ev.leave_one_out_counts(season_era5_rank_da, n_season_members, 'year')
season_pseudo_counts_da = ev.rank_counts(season_pseudo_rank_da, n_season_members, 'year')
season_era5_counts_da.plot.step();

In [ ]:
#(t): 4. Each histogram summarised
season_era5_summary_ds = ev.rank_statistics(season_era5_counts_da)
season_pseudo_summary_ds = ev.rank_statistics(season_pseudo_counts_da)
season_era5_summary_ds

In [ ]:
#(t): Quick look: the pseudo-observations' dispersions (histogram) and ERA5's (black)
season_pseudo_summary_ds['dispersion'].plot.hist(bins=20, color='0.7')
plt.axvline(float(season_era5_summary_ds['dispersion']), color='k');

In [ ]:
#(t): 5. Where ERA5's dispersion falls among the pseudo-observations'
season_dispersion_test_ds = ev.locate(season_pseudo_summary_ds['dispersion'], season_era5_summary_ds['dispersion'])
season_dispersion_test_ds

In [ ]:
#(t): The same through ev.rank_histogram_test, which treats the raw values itself and tests every summary
anatomy_result_ds = ev.rank_histogram_test(historical_point_tree[MODEL]['historical']['tas'].sel(season=SEASON),
                                           era5_point_da.sel(season=SEASON), 'anomaly')
anatomy_result_ds

In [ ]:
#(t): Check: both give the same dispersion test
check_variables = ['era5', 'percentile', 'pvalue', 'verdict']
pd.DataFrame({'step by step': season_dispersion_test_ds[check_variables].to_pandas(),
              'ev.rank_histogram_test': anatomy_result_ds.sel(statistic='dispersion')[check_variables].to_pandas()})

In [ ]:
#(t): Saved for 02b's figure of the method: the anomalies and the test
storage.save(xr.Dataset({'members': season_members_da, 'era5': season_era5_da}), result_path('rank_anatomy_data'))

In [ ]:
storage.save(anatomy_result_ds, result_path('rank_anatomy_test'))

### Every model

The headline histograms pool the four seasons (four times the samples); the scorecard uses one per season.

In [ ]:
#(t): ev.rank_histogram_test on one model's node (it needs each model's own number of members)
@skip_empty
def rank_test(ds, era5_da, treatment, pool_dims, keep_histograms=True):
    return ev.rank_histogram_test(ds['tas'], era5_da, treatment, pool_dims=pool_dims, keep_histograms=keep_histograms)

In [ ]:
#(t): Raw values, the four seasons pooled
raw_ranks_tree = historical_point_tree.map_over_datasets(rank_test, era5_point_da, 'raw', ('year', 'season'))
raw_ranks_tree

In [ ]:
#(t): Anomalies, the four seasons pooled
anomaly_ranks_tree = historical_point_tree.map_over_datasets(rank_test, era5_point_da, 'anomaly', ('year', 'season'))

In [ ]:
#(t): Detrended anomalies, the four seasons pooled
detrended_ranks_tree = historical_point_tree.map_over_datasets(rank_test, era5_point_da, 'detrended', ('year', 'season'))

In [ ]:
#(t): Quick look: MODEL's histogram of anomalies (bars: ERA5; grey: each pseudo-observation's)
anomaly_ranks_ds = anomaly_ranks_tree[MODEL]['historical'].to_dataset()
anomaly_ranks_ds['pseudo_frequency'].plot.line(x='rank', hue='member', add_legend=False, color='0.85')
anomaly_ranks_ds['obs_frequency'].plot.step(color='k');

In [ ]:
#(t): One test per season, for the scorecard (without the histograms): anomalies, and detrended anomalies
anomaly_ranks_seasonal_tree = historical_point_tree.map_over_datasets(rank_test, era5_point_da, 'anomaly', ('year',),
                                                                      False)
detrended_ranks_seasonal_tree = historical_point_tree.map_over_datasets(rank_test, era5_point_da, 'detrended',
                                                                        ('year',), False)

In [ ]:
storage.save(raw_ranks_tree, result_path('raw_ranks'))

In [ ]:
storage.save(anomaly_ranks_tree, result_path('anomaly_ranks'))

In [ ]:
storage.save(detrended_ranks_tree, result_path('detrended_ranks'))

## Every test in one table

Each test's percentile, verdict and whether it could flag anything, without the members, so every model fits in one Dataset on (model, statistic, season): for 02b's scorecard and signal-vs-noise diagram. Both rank tests have a dispersion, so those two are named for their treatment.

In [ ]:
#(t): One node's verdicts only (no members)
@skip_empty
def verdicts_only(ds):
    return ds[['percentile', 'verdict', 'testable']]

In [ ]:
#(t): The moments and the other statistics, every model, on (model, statistic, season)
moment_verdicts_ds = tree_to_dataset(moment_test_tree.map_over_datasets(verdicts_only), dims=['model', 'experiment'])
statistic_verdicts_ds = tree_to_dataset(statistic_test_tree.map_over_datasets(verdicts_only), dims=['model', 'experiment'])

In [ ]:
#(t): The rank trend and dispersion of the anomalies, and the dispersion of the detrended anomalies
anomaly_rank_verdicts_ds = tree_to_dataset(anomaly_ranks_seasonal_tree.map_over_datasets(verdicts_only),
                                           dims=['model', 'experiment'])
anomaly_rank_verdicts_ds = anomaly_rank_verdicts_ds.sel(statistic=['rank_trend', 'dispersion']).drop_vars(['label', 'units'])
anomaly_rank_verdicts_ds = anomaly_rank_verdicts_ds.assign_coords(statistic=['rank_trend', 'dispersion (anomaly)'])

detrended_rank_verdicts_ds = tree_to_dataset(detrended_ranks_seasonal_tree.map_over_datasets(verdicts_only),
                                             dims=['model', 'experiment'])
detrended_rank_verdicts_ds = detrended_rank_verdicts_ds.sel(statistic=['dispersion']).drop_vars(['label', 'units'])
detrended_rank_verdicts_ds = detrended_rank_verdicts_ds.assign_coords(statistic=['dispersion (detrended)'])

In [ ]:
#(t): Everything along one statistic dim
eval_table_by_experiment_ds = xr.concat([moment_verdicts_ds, statistic_verdicts_ds, anomaly_rank_verdicts_ds,
                                         detrended_rank_verdicts_ds], dim='statistic')
eval_table_ds = eval_table_by_experiment_ds.sel(experiment='historical', drop=True)
#(c): tree_to_dataset leaves the model names as Python objects, which xarray will not put on an axis
eval_table_ds = eval_table_ds.assign_coords(model=eval_table_ds['model'].astype(str))
eval_table_ds

In [ ]:
#(t): Quick look: ERA5's percentile among each model's members (50: in the middle; 0 or 100: beyond every member)
eval_table_ds['percentile'].sel(season=SEASONS).plot(x='season', y='model', col='statistic', col_wrap=5, vmin=0,
                                                     vmax=100, cmap='RdBu_r');

In [ ]:
storage.save(eval_table_ds, result_path('eval_table'))

## The tests on toy ensembles with a known answer

Before trusting the tests on real data: synthetic ensembles where we *know* what is wrong (`ev.simulate_ensemble`: 50 members and a pseudo-ERA5, 1979–2013, four seasons, each series trend × decades + noise × N(0, 1)). Pseudo-ERA5 always has a trend of 0.3 per decade and noise 1; each scenario of `ev.SCENARIOS` changes one thing about the model. 02b draws them.

In [ ]:
#(t): A toy ensemble and pseudo-ERA5 for each scenario, side by side on a scenario dim (all have 50 members)
demo_datasets = []
for settings in ev.SCENARIOS.values():
    demo_members_da, demo_era5_da = ev.simulate_ensemble(seed=3, **settings)
    demo_datasets.append(xr.Dataset({'members': demo_members_da, 'era5': demo_era5_da}))
demo_ds = xr.concat(demo_datasets, dim=xr.Variable('scenario', list(ev.SCENARIOS)))
demo_ds

In [ ]:
#(t): Quick look: each scenario's members (grey) and pseudo-ERA5 (black) in DJF
fg = demo_ds['members'].sel(season='DJF').plot.line(x='year', hue='member', col='scenario', add_legend=False,
                                                    color='0.7', lw=0.4)
for ax, scenario in zip(fg.axs.flat, demo_ds.scenario.values):
    demo_ds['era5'].sel(season='DJF', scenario=scenario).plot(ax=ax, color='k')

In [ ]:
#(t): The detrended values
demo_detrended_ds = demo_ds.map(ev.detrend)

In [ ]:
#(t): Four statistics of each, and where pseudo-ERA5's falls: the mean and trend of the raw values, the std and width
#(t): of the detrended values
demo_mean_ds = demo_ds.mean('year')
demo_trend_ds = demo_ds.map(ev.trend)
demo_std_ds = demo_detrended_ds.std('year', ddof=1)
demo_width_ds = demo_detrended_ds.map(ev.quantile_width)

In [ ]:
demo_tests_ds = xr.concat([ev.locate(demo_mean_ds['members'], demo_mean_ds['era5']),
                           ev.locate(demo_trend_ds['members'], demo_trend_ds['era5']),
                           ev.locate(demo_std_ds['members'], demo_std_ds['era5']),
                           ev.locate(demo_width_ds['members'], demo_width_ds['era5'])],
                          dim=xr.Variable('statistic', ['mean', 'trend', 'std', 'width']))
demo_tests_ds['percentile'].sel(season='DJF').plot(x='scenario', y='statistic', vmin=0, vmax=100, cmap='RdBu_r');

In [ ]:
#(t): The rank-histogram tests, the four seasons pooled: anomalies, and detrended anomalies
demo_anomaly_ranks_ds = ev.rank_histogram_test(demo_ds['members'], demo_ds['era5'], 'anomaly',
                                               pool_dims=('year', 'season'))
demo_detrended_ranks_ds = ev.rank_histogram_test(demo_ds['members'], demo_ds['era5'], 'detrended',
                                                 pool_dims=('year', 'season'))

In [ ]:
#(t): The plume of the raw values: the members' quantiles each year, and where pseudo-ERA5 is
demo_plume_ds = ev.ensemble_plume(demo_ds['members'], demo_ds['era5'], 'raw')

In [ ]:
storage.save(demo_tests_ds.drop_dims('member'), result_path('demo_tests'))

In [ ]:
storage.save(demo_anomaly_ranks_ds, result_path('demo_anomaly_ranks'))

In [ ]:
storage.save(demo_detrended_ranks_ds, result_path('demo_detrended_ranks'))

In [ ]:
storage.save(demo_plume_ds, result_path('demo_plume'))

## 7. Maps: the moment test at every grid point

The moment test of the point, at every grid point of every model. It takes minutes, so it saves its result; the *reopen* cell is for next time. Then the **field test** (`ev.field_test`): a perfect model is flagged at about 10% of grid points by chance, and neighbouring points are correlated, so the question is whether ERA5 is flagged over *more* of the map than a perfect model would be. Each member in turn plays ERA5 against the other N − 1, and the areas they are flagged over show what a perfect model gives (Livezey & Chen, 1983, with the members as the null).

### Calculate (slow; saves the result)

In [ ]:
#(t): The moment test at every grid point of one model, by the same steps as at the point, then the field test
#(c): One model's 1979-2013 field (~100 MB) is loaded at a time. The members are dropped once tested, so models of
#(c): any ensemble size stack on one grid
def map_moment_test(ds):
    members_da = ds['tas'].load()
    era5_da = era5_seasonal_da.sel(year=YEARS)
    members_detrended_da = ev.detrend(members_da)
    era5_detrended_da = ev.detrend(era5_da)

    mean_test_ds = ev.locate(members_da.mean('year'), era5_da.mean('year'))
    std_test_ds = ev.locate(members_detrended_da.std('year', ddof=1), era5_detrended_da.std('year', ddof=1))
    skewness_test_ds = ev.locate(ev.skewness(members_detrended_da), ev.skewness(era5_detrended_da))
    kurtosis_test_ds = ev.locate(ev.excess_kurtosis(members_detrended_da), ev.excess_kurtosis(era5_detrended_da))
    moments_ds = xr.concat([mean_test_ds, std_test_ds, skewness_test_ds, kurtosis_test_ds],
                           dim=xr.Variable('statistic', ['mean', 'std', 'skewness', 'kurtosis']))

    field_test_ds = ev.field_test(moments_ds)
    return xr.merge([moments_ds, field_test_ds]).drop_dims('member')

In [ ]:
%%time
#(t): For every model, on (model, statistic, season, lat, lon)
eval_map_by_experiment_ds = datatree.reduce_to_dataset(lesfmip_obs_tree.match('*/historical'), map_moment_test,
                                                       coords='minimal', compat='override')
eval_map_table_ds = eval_map_by_experiment_ds.sel(experiment='historical', drop=True)

In [ ]:
storage.save(eval_map_table_ds, result_path('eval_map_table'))

### Or reopen the saved result

In [ ]:
eval_map_table_ds = storage.open_dataset(result_path('eval_map_table'), load=True)

### What was made

In [ ]:
eval_map_table_ds

In [ ]:
#(t): Check: at the point the maps give the point's moment test (True if every percentile agrees)
moment_percentile_ds = tree_to_dataset(moment_test_tree.map_over_datasets(verdicts_only), dims=['model', 'experiment'])
moment_percentile_da = moment_percentile_ds['percentile'].sel(experiment='historical', drop=True)
map_point_percentile_da = eval_map_table_ds['percentile'].sel(POINT).sel(model=moment_percentile_da.model)
bool(np.allclose(map_point_percentile_da.transpose(*moment_percentile_da.dims), moment_percentile_da, equal_nan=True))

In [ ]:
#(t): Quick look: ERA5's percentile among each model's members for the standard deviation
eval_map_table_ds['percentile'].sel(statistic='std').plot(x='lon', y='lat', row='model', col='season', vmin=0, vmax=100,
                                                          cmap='RdBu_r');

## 8. Where each model is adequate: the spatial evaluation of Suarez-Gutierrez et al. (2021)

The paper's grid-point evaluation (their sections 2.2.2 and 3.2–3.4), at every grid point and season, by `ev.spatial_evaluation`: each year, is ERA5 (in anomalies) below every member, above every member, or in their central 75%? A perfect model puts ERA5 beyond each edge in 100/(N + 1)% of years and in the middle about 75% of the time; the paper flags ≥ 10% below, ≥ 10% above or > 80% in the middle. When the problem years fall (1979–1996 or 1997–2013) then says whether the forced response or the variability is wrong. 02b draws the maps.

### Calculate (slow; saves the result)

In [ ]:
#(t): The paper's thresholds (10% / 10% / 80%), and the two halves of the record
SPATIAL_THRESHOLDS = 'fixed'
SPATIAL_PERIODS = {'early': slice(1979, 1996), 'late': slice(1997, None)}

In [ ]:
#(t): One model: the whole record, each half, and whether a problem is spread evenly over the two (the variability)
#(t): or not (the forced response)
def spatial_evaluation(ds):
    members_da = ds['tas'].load()
    era5_da = era5_seasonal_da.sel(year=YEARS)
    whole_ds = ev.spatial_evaluation(members_da, era5_da, thresholds=SPATIAL_THRESHOLDS)
    early_ds = ev.spatial_evaluation(members_da, era5_da, period=SPATIAL_PERIODS['early'], thresholds=SPATIAL_THRESHOLDS)
    late_ds = ev.spatial_evaluation(members_da, era5_da, period=SPATIAL_PERIODS['late'], thresholds=SPATIAL_THRESHOLDS)
    return whole_ds.assign(period_diagnosis=ev.period_comparison(whole_ds, early_ds, late_ds))

In [ ]:
%%time
spatial_by_experiment_ds = datatree.reduce_to_dataset(lesfmip_obs_tree.match('*/historical'), spatial_evaluation,
                                                      coords='minimal', compat='override')
spatial_table_ds = spatial_by_experiment_ds.sel(experiment='historical', drop=True)

In [ ]:
storage.save(spatial_table_ds, result_path('spatial_table'))

### Or reopen the saved result

In [ ]:
spatial_table_ds = storage.open_dataset(result_path('spatial_table'), load=True)

### What was made

In [ ]:
spatial_table_ds

In [ ]:
#(t): Quick look: the % of years ERA5 is below every member
spatial_table_ds['below'].plot(x='lon', y='lat', row='model', col='season', vmin=0, vmax=30);

## 9. The regional evaluation, saved for notebook 07

A regional mean averages away much of a grid point's noise, which is why the paper's verification is regional:

- ERA5's regional means: `era5_regional`;
- the rank-histogram test of each region's mean, every model and experiment, anomalies and detrended: `regional_ranks`;
- ERA5's 1979–2013 trend among the members of every experiment: `regional_consistency`;
- ERA5's trend map, and each model's ensemble-mean trend in every experiment: `trend_maps`.

In [ ]:
#(t): The regions (observed_change.REGIONS), as in part 2
oc.REGIONS

In [ ]:
#(t): ERA5's regional means, on (region, year, season)
era5_regional_da = oc.regional_means(era5_seasonal_da).rename(VARIABLE)
era5_regional_da

In [ ]:
storage.save(era5_regional_da, result_path('era5_regional'))

In [ ]:
%%time
#(t): Every member's regional means over the ERA5 years, every model and experiment
regional_obs_tree = oc.regional_means(lesfmip_obs_tree).compute()
regional_obs_tree

In [ ]:
#(t): Quick look: ERA5 (black) among MODEL's historical members (grey) in the first region, every season
regional_ensemble_da = regional_obs_tree[MODEL]['historical'][VARIABLE].isel(region=0)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharex=True)
for ax, season in zip(axes, ev.SEASONS):
    ax.plot(regional_ensemble_da.year, regional_ensemble_da.sel(season=season).transpose('year', 'member'), color='0.7', lw=0.5)
    ax.plot(era5_regional_da.year, era5_regional_da.isel(region=0).sel(season=season), color='k')
    ax.set_title(f"{str(regional_ensemble_da.region.values)}, {season}")

In [ ]:
#(t): ev.rank_histogram_summary on one node of regional_obs_tree, against ERA5's regional means over YEARS
@skip_empty
def regional_rank_summary(ds, treatment):
    return ev.rank_histogram_summary(ds['tas'], era5_regional_da.sel(year=YEARS), treatment)

In [ ]:
%%time
#(t): Every model and experiment, regional means, anomalies
regional_ranks_anomaly_tree = regional_obs_tree.map_over_datasets(regional_rank_summary, 'anomaly')
regional_ranks_anomaly_ds = tree_to_dataset(regional_ranks_anomaly_tree, dims=['model', 'experiment'],
                                            coords='minimal', compat='override')
regional_ranks_anomaly_ds

In [ ]:
%%time
#(t): ... and detrended anomalies
regional_ranks_detrended_tree = regional_obs_tree.map_over_datasets(regional_rank_summary, 'detrended')
regional_ranks_detrended_ds = tree_to_dataset(regional_ranks_detrended_tree, dims=['model', 'experiment'],
                                              coords='minimal', compat='override')

In [ ]:
#(t): The two along a treatment dim
regional_ranks_ds = xr.concat([regional_ranks_anomaly_ds.drop_vars('treatment'),
                               regional_ranks_detrended_ds.drop_vars('treatment')],
                              dim=xr.Variable('treatment', ['anomaly', 'detrended']))
regional_ranks_ds

In [ ]:
storage.save(regional_ranks_ds, result_path('regional_ranks'))

In [ ]:
#(t): Quick look: MODEL's historical histograms (anomalies) in the first region, ERA5 against the perfect-model range
regional_ranks_ds[['obs_frequency', 'pseudo_lower', 'pseudo_upper']].sel(
    model=MODEL, experiment='historical', treatment='anomaly').isel(region=0).to_dataarray('line').plot.line(
    x='rank', hue='line', col='season');

In [ ]:
#(t): ERA5's regional trend among the members of one node (the members dropped)
@skip_empty
def regional_trend_test(ds):
    members_trend_da = ev.trend(ds['tas'])
    era5_trend_da = ev.trend(era5_regional_da.sel(year=YEARS))
    return ev.locate(members_trend_da, era5_trend_da).drop_dims('member')

In [ ]:
%%time
#(t): Every model and experiment
regional_trend_tree = regional_obs_tree.map_over_datasets(regional_trend_test)

In [ ]:
#(t): On (statistic, model, experiment, region, season), the statistic named, as notebook 07 opens it
regional_trend_ds = tree_to_dataset(regional_trend_tree, dims=['model', 'experiment'])
regional_consistency_ds = regional_trend_ds.expand_dims(statistic=['trend'])
regional_consistency_ds

In [ ]:
storage.save(regional_consistency_ds.drop_vars(['label', 'units', 'treatment'], errors='ignore'),
             result_path('regional_consistency'))

In [ ]:
#(t): Quick look: ERA5's trend percentile among each experiment's members, MODEL, every region and season
consistency_percentile_da = regional_consistency_ds['percentile'].sel(model=MODEL, statistic='trend', season=SEASONS)
consistency_percentile_da.assign_coords(experiment=consistency_percentile_da['experiment'].astype(str)).plot(
    x='season', y='experiment', col='region', vmin=0, vmax=100, cmap='RdBu_r');

In [ ]:
#(t): The years ERA5 and the models share (LESFMIP ends with DJF 2013)
trend_years_arr = np.intersect1d(era5_seasonal_da['year'], lesfmip_obs_tree[MODEL]['historical']['year'])
trend_years_arr[[0, -1]]

In [ ]:
#(t): ERA5's trend over them at every grid point (per decade)
era5_trend_da = ev.trend(era5_seasonal_da.sel(year=trend_years_arr))
era5_trend_da

In [ ]:
%%time
#(t): Every model's ensemble-mean trend over the same years, every experiment
model_trend_ds = datatree.reduce_to_dataset(lesfmip_obs_tree.sel(year=trend_years_arr),
                                            lambda ds: ds.mean('member').map(ev.trend)).compute()
model_trend_ds

In [ ]:
#(t): The trend maps in one Dataset, laid out as notebook 07 opens them
trend_maps_ds = xr.Dataset({'era5_trend': era5_trend_da, 'model_trend': model_trend_ds[VARIABLE]})
trend_maps_ds

In [ ]:
storage.save(trend_maps_ds, result_path('trend_maps'))

In [ ]:
#(t): Quick look: ERA5's trend, every season
trend_maps_ds['era5_trend'].sel(season=SEASONS).plot(x='lon', y='lat', col='season', cmap='RdBu_r', center=0);

# Part 2: observed change

The ERA5 evaluation asked whether ERA5 could be a member of the historical ensemble. Attribution asks more of the observations:

1. **Detection.** Could the observed change have happened without human influence? ERA5 outside the **hist-nat** members' range says no.
2. **Consistency.** Which experiments could have produced it: historical, or a single forcing on its own?
3. **Records.** With no change, year *n* is its record's warmest so far with probability 1/*n*. Warming brings more record highs and fewer record lows.
4. **Scaling factors.** How much of each model's forced response is in the observations.

Mostly in regional means (`oc.REGIONS`), which average away much of a grid point's noise.

In [ ]:
#(t): The region of the one-region figures, and the model of the one-model figures
OBS_REGION = 'Antarctica (90–65°S)'
OBS_MODEL = 'CanESM5'
oc.REGIONS

In [ ]:
#(t): The experiments every model has
OBS_EXPERIMENTS = [experiment for experiment in EXPERIMENTS
                   if all(experiment in model_node.children for model_node in lesfmip_season_tree.children.values())]
OBS_EXPERIMENTS

## 1. Which experiments is the observed change consistent with?

ERA5's 1979–2013 trend, and its S/N against itself, among each experiment's members, in `OBS_REGION`. ERA5 outside **hist-nat** is a detection of forced change; ERA5 inside **historical** is consistency.

In [ ]:
#(t): Every member's regional mean in OBS_REGION, every model and experiment (section 9's regional_obs_tree)
obs_region_tree = regional_obs_tree.sel(region=OBS_REGION)

In [ ]:
#(t): ERA5's, over YEARS
era5_obs_region_da = era5_regional_da.sel(region=OBS_REGION, year=YEARS)

In [ ]:
#(t): Quick look: MODEL's historical (grey) and hist-nat (green) members, and ERA5 (black), in JJA
obs_region_tree[OBS_MODEL]['historical']['tas'].sel(season='JJA').plot.line(x='year', hue='member', add_legend=False,
                                                                             color='0.7', lw=0.5)
obs_region_tree[OBS_MODEL]['hist-nat']['tas'].sel(season='JJA').plot.line(x='year', hue='member', add_legend=False,
                                                                           color='tab:green', lw=0.3)
era5_obs_region_da.sel(season='JJA').plot(color='k', lw=2);

In [ ]:
#(t): The trend of every member, and ERA5's
region_trend_tree = obs_region_tree.map_over_datasets(skip_empty(ev.trend))
era5_region_trend_da = ev.trend(era5_obs_region_da)

In [ ]:
region_trend_test_tree = region_trend_tree.map_over_datasets(locate_era5, era5_region_trend_da)

In [ ]:
#(t): The S/N against itself, the same way
region_emergence_tree = obs_region_tree.map_over_datasets(skip_empty(ev.self_signal_to_noise))
era5_region_emergence_da = ev.self_signal_to_noise(era5_obs_region_da)
region_emergence_test_tree = region_emergence_tree.map_over_datasets(locate_era5, era5_region_emergence_da)

In [ ]:
#(t): One node's two tests along a statistic dim
@skip_empty
def trend_and_emergence(trend_ds, emergence_ds):
    return xr.concat([trend_ds, emergence_ds], dim=xr.Variable('statistic', ['trend', 'emergence']))

In [ ]:
region_tests_tree = xr.map_over_datasets(trend_and_emergence, region_trend_test_tree, region_emergence_test_tree)
region_tests_tree

In [ ]:
#(t): Quick look: ERA5's trend percentile among each experiment's members, every model
region_trend_verdicts_ds = tree_to_dataset(region_trend_test_tree.map_over_datasets(verdicts_only),
                                           dims=['model', 'experiment'])
region_trend_verdicts_ds = region_trend_verdicts_ds.assign_coords(model=region_trend_verdicts_ds['model'].astype(str),
                                                                  experiment=region_trend_verdicts_ds['experiment'].astype(str))
region_trend_verdicts_ds['percentile'].sel(season=SEASONS).plot(x='season', y='model', col='experiment', vmin=0,
                                                                vmax=100, cmap='RdBu_r');

In [ ]:
storage.save(region_tests_tree, result_path('region_tests'))

## 2. Record rates

Cumulative record highs and lows through 1979–2013, the four seasons pooled so there are enough records to compare, for ERA5 and the hist-nat and historical members (`oc.record_curves`).

In [ ]:
#(t): In OBS_REGION
record_curves_ds = oc.record_curves({'hist-nat': obs_region_tree[OBS_MODEL]['hist-nat']['tas'],
                                     'historical': obs_region_tree[OBS_MODEL]['historical']['tas']}, era5_obs_region_da)
record_curves_ds

In [ ]:
#(t): At the point (much noisier than a regional mean)
point_curves_ds = oc.record_curves({'hist-nat': histnat_point_tree[OBS_MODEL]['hist-nat']['tas'],
                                    'historical': historical_point_tree[OBS_MODEL]['historical']['tas']}, era5_point_da)

In [ ]:
#(t): Quick look: the record highs, ERA5 (black) and each historical member (grey)
record_curves_ds['members'].sel(experiment='historical', kind='highs').plot.line(x='year', hue='member',
                                                                                 add_legend=False, color='0.7')
record_curves_ds['obs'].sel(kind='highs').plot(color='k');

In [ ]:
storage.save(record_curves_ds, result_path('record_curves'))

In [ ]:
storage.save(point_curves_ds, result_path('point_record_curves'))

## 3. Detection maps

The trend test at every grid point, against hist-nat and against historical, sorted into four classes (`oc.DETECTION_CLASSES`): detected and consistent (outside hist-nat, inside historical), not detected and consistent, detected and inconsistent, not detected and inconsistent. It takes a few minutes, so it saves its result.

### Calculate (slow; saves the result)

In [ ]:
#(t): ERA5's trend among one experiment's members at every grid point (one model's experiment loaded at a time;
#(t): the members dropped)
def map_trend_test(ds):
    members_trend_da = ev.trend(ds['tas'].load())
    era5_trend_da = ev.trend(era5_seasonal_da.sel(year=YEARS))
    return ev.locate(members_trend_da, era5_trend_da).drop_dims('member')

In [ ]:
%%time
#(t): For historical and hist-nat of every model, on (model, experiment, season, lat, lon)
detection_tree = lesfmip_obs_tree.filter(lambda node: node.name in ('historical', 'hist-nat'))
map_trend_tests_ds = datatree.reduce_to_dataset(detection_tree, map_trend_test)

In [ ]:
#(t): Each grid point's class
detection_class_da = oc.detection_class(map_trend_tests_ds.sel(experiment='historical'),
                                        map_trend_tests_ds.sel(experiment='hist-nat'))
detection_class_da.plot(x='lon', y='lat', row='model', col='season');

In [ ]:
#(t): The classes and ERA5's percentile among each experiment's members, as notebook 07 opens them
detections_ds = xr.Dataset({
    'detection_class': detection_class_da,
    'historical_percentile': map_trend_tests_ds['percentile'].sel(experiment='historical', drop=True),
    'hist-nat_percentile': map_trend_tests_ds['percentile'].sel(experiment='hist-nat', drop=True),
})
detections_ds

In [ ]:
storage.save(detections_ds, result_path('detections'))

## 4. Scaling factors

A regression of ERA5 on each model's forced responses, over every year and season of 1979–2014, in each region (`oc.scaling_factors`): ANT + NAT (anthropogenic, historical − hist-nat, and natural), and GHG + AER + O3 + NAT. The uncertainty comes from a perfect-model test: each historical member in turn plays the observations, regressed on fingerprints that leave it out.

In [ ]:
%%time
#(t): Every region and model, for each set of signals, along (region, model, signal)
scaling_datasets = {}
for signals in oc.SIGNAL_SETS:
    rows = []
    for region in oc.REGIONS:
        for model in regional_obs_tree.children:
            experiments = {experiment: regional_obs_tree[model][experiment]['tas'].sel(region=region)
                           for experiment in OBS_EXPERIMENTS}
            result_ds = oc.scaling_factors(experiments, era5_regional_da.sel(region=region), signals)
            rows.append(result_ds.drop_vars('pseudo_beta').expand_dims(region=[region], model=[model]))
    scaling_datasets[signals] = xr.combine_by_coords(rows)
scaling_datasets['ANT + NAT']

In [ ]:
storage.save(scaling_datasets['ANT + NAT'], result_path('scaling_ant_nat'))

In [ ]:
storage.save(scaling_datasets['GHG + AER + O3 + NAT'], result_path('scaling_single_forcings'))